In [1]:
%%bash
git clone https://github.com/kohya-ss/sd-scripts.git /kaggle/working/sd-scripts
cd /kaggle/working/sd-scripts

pip install --upgrade pip
pip install -r requirements.txt
pip install xformers bitsandbytes accelerate prodigyopt

accelerate config default

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Ignoring tensorboardX: markers 'platform_machine == "ARM64"' don't match your environment
Obtaining file:///kaggle/working/sd-scripts (from -r requirements.txt (line 58))
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 113.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Cloning into '/kaggle/working/sd-scripts'...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
bigframes 2.48.0 requires rich<14,>=12.4.4, but you have rich 14.1.0 which is incompatible.


In [2]:
import os
import glob
from PIL import Image

input_dir = "/kaggle/input/datasets/anindoakb/guinevere-lora"
output_dir = "/kaggle/working/train_guinevere/img/40_guinevere_mlbb"
os.makedirs(output_dir, exist_ok=True)

image_paths = glob.glob(os.path.join(input_dir, "*.*"))

for idx, img_path in enumerate(image_paths):
    try:
        img = Image.open(img_path).convert("RGB")
        
        # Save as high-quality JPG to prevent tensor errors
        base_out = os.path.join(output_dir, f"img_{idx:04d}.jpg")
        img.save(base_out, "JPEG", quality=100)
        
        # Augment with horizontal flip for 360-degree pose learning
        img_flip = img.transpose(Image.FLIP_LEFT_RIGHT)
        flip_out = os.path.join(output_dir, f"img_{idx:04d}_aug_flip.jpg")
        img_flip.save(flip_out, "JPEG", quality=100)
    except Exception as e:
        print(f"Skipping {img_path} due to error: {e}")

print(f"Dataset synthesis complete. Images: {len(os.listdir(output_dir))}")

Dataset synthesis complete. Images: 136


In [3]:
import os

prompt_file = "/kaggle/working/prompt.txt"

prompts = [
    "guinevere_mlbb, 1girl, fighting stance, glowing magic, intense gaze, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, jumping mid-air, dynamic action scene, flowing hair, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, sitting elegantly on a throne, regal posture, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, looking over shoulder, seductive smile, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, crouching low, stealthy, ready to pounce, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, casting magic, ethereal glow, magical circle, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, running towards the camera, breathless, action blur, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, lying on the grass, peaceful expression, looking at sky, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, back to camera, turning head, flowing dress, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, mid-kick, martial arts pose, dynamic composition, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, floating in water, ethereal dress, serene face, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, kneeling in prayer, hands clasped, divine light, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, pointing finger forward, commanding presence, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, leaning against a wall, casual yet elegant, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, doing a pirouette, ballet pose, twirling skirt, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, upside down, falling through the sky, wind in hair, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, drawing a sword, intense combat readiness, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, hugging knees to chest, vulnerable, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, reaching hand out, inviting gesture, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, profile view, side angle, gazing into the distance, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face"
]

flags = "--w 768 --h 1024 --l 7.5 --s 30"

with open(prompt_file, "w", encoding="utf-8") as f:
    for prompt in prompts:
        f.write(f"{prompt} {flags}\n")

In [4]:
import os
import glob
import shutil

# 1. Force single GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 2. Wipe the old training directory to force a clean reset
base_train_dir = "/kaggle/working/train_guinevere"
if os.path.exists(base_train_dir):
    shutil.rmtree(base_train_dir)

# 3. Setup folders for Guinevere with EXACTLY 10 repeats 
input_dir = "/kaggle/input/datasets/anindoakb/guinevere-lora"
img_dir = os.path.join(base_train_dir, "img", "10_guinevere_mlbb") 
output_dir = "/kaggle/working/output_guinevere"

os.makedirs(img_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# 4. Copy and convert images
print("Preparing Guinevere images for fast 10-repeat training...")
extensions = ("*.png", "*.jpg", "*.jpeg", "*.webp")
for ext in extensions:
    for img_path in glob.glob(os.path.join(input_dir, "**", ext), recursive=True):
        filename = os.path.basename(img_path)
        from PIL import Image
        try:
            with Image.open(img_path) as img:
                rgb_img = img.convert("RGB")
                safe_filename = os.path.splitext(filename)[0] + ".jpg"
                rgb_img.save(os.path.join(img_dir, safe_filename), quality=100)
        except Exception as e:
            pass
print(f"Copied {len(os.listdir(img_dir))} images.")

# 5. Start the optimized 5-Epoch Training
print("\nStarting LoRA Training...")
!cd /kaggle/working/sd-scripts && accelerate launch \
  --num_processes=1 \
  --mixed_precision="fp16" \
  train_network.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/working/train_guinevere/img" \
  --output_dir="/kaggle/working/output_guinevere" \
  --output_name="guinevere_lora" \
  --network_module="networks.lora" \
  --network_dim=256 \
  --network_alpha=128 \
  --optimizer_type="Prodigy" \
  --learning_rate=1.0 \
  --unet_lr=1.0 \
  --text_encoder_lr=1.0 \
  --lr_scheduler="cosine" \
  --optimizer_args weight_decay=0.01 d_coef=2 use_bias_correction=True safeguard_warmup=True \
  --min_snr_gamma=5.0 \
  --noise_offset=0.1 \
  --multires_noise_iterations=6 \
  --multires_noise_discount=0.3 \
  --network_dropout=0.1 \
  --scale_weight_norms=1.0 \
  --color_aug \
  --random_crop \
  --mixed_precision="fp16" \
  --save_precision="fp16" \
  --xformers \
  --gradient_checkpointing \
  --max_train_epochs=5 \
  --save_every_n_epochs=1 \
  --train_batch_size=2 \
  --resolution="512,512" \
  --save_model_as="safetensors"

Preparing Guinevere images for fast 10-repeat training...
Copied 68 images.

Starting LoRA Training...
/kaggle/working/sd-scripts/library/strategy_base.py:107: SyntaxWarning: invalid escape sequence '\('
  \( - literal character '('
/kaggle/working/sd-scripts/library/custom_train_functions.py:174: SyntaxWarning: invalid escape sequence '\('
  \( - literal character '('
/kaggle/working/sd-scripts/library/lpw_stable_diffusion.py:70: SyntaxWarning: invalid escape sequence '\('
  \( - literal character '('
2026-10-03 09:08:59 INFO     Using v1 tokenizer                ]8;id=546326;file:///kaggle/working/sd-scripts/library/strategy_sd.py\strategy_sd.py]8;;\:]8;id=391948;file:///kaggle/working/sd-scripts/library/strategy_sd.py#34\34]8;;\
2026-10-03 09:09:00 INFO     HTTP Request: GET                   ]8;id=399820;file:///usr/local/lib/python3.13/dist-packages/httpx/_client.py\_client.py]8;;\:]8;id=962738;file:///usr/local/lib/python3.13/dist-packages/httpx/_client.py#1025\1025

In [5]:
import shutil
import os

output_folder = "/kaggle/working/output_guinevere"
archive_name = "/kaggle/working/guinevere_lora_outputs"

if os.path.exists(output_folder):
    shutil.make_archive(archive_name, 'zip', output_folder)
    print(f"Export complete. Archive saved to {archive_name}.zip")
else:
    print(f"Folder {output_folder} not found. Ensure training finished successfully.")

Export complete. Archive saved to /kaggle/working/guinevere_lora_outputs.zip
